# Agent Evaluation with Google ADK — Workshop (Colab)

A hands-on tour of evaluating agents properly: **trajectory evaluation**,
**LLM-as-a-Judge**, and **eval as an engineering discipline (CI)**.

### Why we're running in Colab
Some corporate networks block the Google AI endpoints (`generativelanguage.googleapis.com`,
`aiplatform.googleapis.com`). In Colab the model calls run from **Google's network, not your laptop**,
so your workshop API key works here even when it's blocked on the office network.

### Sign-in into Colab
Sign to Google colab's using your Google account(Do not use your company's Google account to sign in)

**How to use this notebook:** make your own copy (File → Save a copy in Drive), then run the
cells top to bottom. Each section builds on the previous one.

## 0. Setup

Install ADK (with the `eval` extra) and set your API key (Ignore the pandas version conflict message)

In [ ]:
!pip install -q "google-adk[eval]"

In [ ]:
import os, getpass, logging, warnings

# Keep the notebook output clean: silence ADK experimental-feature warnings and
# the google_genai "non-text parts" info warning (it fires whenever a response
# contains tool calls, which is normal for us).
warnings.filterwarnings("ignore")
logging.getLogger("google_genai").setLevel(logging.ERROR)
logging.getLogger("google.adk").setLevel(logging.ERROR)

# The judge & agent must reach Gemini via the API key (NOT Vertex).
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"

# In Colab you can the store keys in the Secrets panel — the key icon on the left.
# Store the provided key as GOOGLE_API_KEY
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    if not os.environ.get("GOOGLE_API_KEY"):
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Do not paste your workshop API key")

print("API key set:", bool(os.environ.get("GOOGLE_API_KEY")))

## 1. The agent we'll evaluate

A **customer-support agent that handles refund requests using tools**.

### Why Tools Exist

Tools exist to give AI models **capabilities beyond text generation**.

1. Overcome Knowledge Limits: Models cannot access private databases or real-time data without
tools.
2. Enable Action: Tools allow models to perform actions like sending emails or processing payments.
3. Ensure Accuracy: Tools handle precise calculations and logic that prevent model hallucination

### Types of ADK Tools

The Agent Development Kit categorizes tools into three primary groups:

1. Function Tools: Custom code written by developers to connect the AI to specific internal APIs.
2. Built-in Tools: Ready-to-use native features like Google Search and code execution sandboxes
3. Third-Party & MCP Tools: Standard integrations that connect agents to external platforms like GitHub or SQL databases

**Our agent has three tools that its uses to answer a question about refund:**

| Tool | Type | Role |
|---|---|---|
| `get_order_status(order_id)` | read | find the order; can fail if unknown |
| `get_refund_policy(category)` | read | the eligibility check |
| `issue_refund(order_id, amount)` | **write** | the risky action — last, and only if eligible |

The correct trajectory is always: **look up the order → check the policy → only if eligible, issue the refund.**

The next cells write the agent to disk as a package (`refund_agent/`), exactly like the repo,
so we can evaluate it with `adk eval`.

> Why write files at all? `adk eval` and `AgentEvaluator` operate on files on disk — they read an
> agent *package* (`refund_agent/`) and JSON eval sets by path. Writing them out lets the notebook
> run the **real** ADK tooling, identical to the repo.

In [ ]:
import os
# %%writefile does NOT create folders, so make them first.
for d in ["refund_agent", "eval"]:
    os.makedirs(d, exist_ok=True)
print("folders ready:", [d for d in ["refund_agent", "eval"] if os.path.isdir(d)])

In [ ]:
%%writefile refund_agent/data.py
"""Seed data for the refund-support agent.

This is a deliberately tiny, in-memory "database".

The five orders are hand-picked to exercise every branch the agent can take:

    A1001  valid, within the refund window        -> refund SHOULD be issued
    A1002  valid, but PAST the refund window       -> refund must be DECLINED
    A1003  valid, item arrived damaged             -> refund SHOULD be issued
    A1004  (does not exist)                         -> agent must ask for a valid id
    A1005  valid, but ALREADY refunded             -> agent must NOT double-refund

"today" is pinned so the "within window" math never drifts between runs.
"""

from datetime import date

# Pinned reference date so policy-window math is deterministic across runs.
TODAY = date(2026, 7, 2)

# Refund policy per product category: how many days after the order date a
# refund can still be issued, plus any always-eligible conditions.
REFUND_POLICY = {
    "electronics": {
        "window_days": 30,
        "always_eligible_if": ["damaged", "defective"],
        "notes": "Electronics are refundable within 30 days, or any time if damaged or defective.",
    },
    "clothing": {
        "window_days": 60,
        "always_eligible_if": ["damaged"],
        "notes": "Clothing is refundable within 60 days, or any time if it arrived damaged.",
    },
    "default": {
        "window_days": 30,
        "always_eligible_if": ["damaged", "defective"],
        "notes": "Standard policy: refundable within 30 days, or any time if damaged or defective.",
    },
}

# The "orders" table.
ORDERS = {
    "A1001": {
        "order_id": "A1001",
        "item": "Wireless headphones",
        "category": "electronics",
        "price": 79.99,
        "order_date": "2026-06-25",  # within 30-day window
        "status": "delivered",
        "refunded": False,
    },
    "A1002": {
        "order_id": "A1002",
        "item": "Bluetooth speaker",
        "category": "electronics",
        "price": 49.99,
        "order_date": "2026-04-01",  # OUTSIDE 30-day window
        "status": "delivered",
        "refunded": False,
    },
    "A1003": {
        "order_id": "A1003",
        "item": "Ceramic vase",
        "category": "home",
        "price": 49.99,
        "order_date": "2026-06-21",  # within window AND will be reported damaged
        "status": "delivered",
        "refunded": False,
    },
    "A1005": {
        "order_id": "A1005",
        "item": "Running shoes",
        "category": "clothing",
        "price": 119.99,
        "order_date": "2026-06-20",  # within window, but ALREADY refunded
        "status": "delivered",
        "refunded": True,
    },
    # NOTE: A1004 is intentionally absent so we can test the "order not found" path.
}


In [ ]:
%%writefile refund_agent/tools.py
"""The three tools the refund agent can call.

Design notes for the workshop:

* All three are plain Python functions wrapped as ADK FunctionTools.

  The "correct" trajectory is: look up the order -> check the policy ->
  (only if eligible) issue the refund.
"""

from datetime import date

from google.adk.tools import FunctionTool

from .data import ORDERS, REFUND_POLICY, TODAY


def get_order_status(order_id: str) -> dict:
    """Look up an order by its ID.

    Args:
        order_id: The customer's order identifier, e.g. "A1001".

    Returns:
        A dict describing the order. If the order does not exist, returns
        {"found": False, ...} so the agent can ask for a valid ID instead of
        guessing.
    """
    order = ORDERS.get(order_id.strip().upper())
    if order is None:
        return {
            "found": False,
            "order_id": order_id,
            "message": f"No order found with ID '{order_id}'.",
        }
    return {
        "found": True,
        "order_id": order["order_id"],
        "item": order["item"],
        "category": order["category"],
        "price": order["price"],
        "order_date": order["order_date"],
        "status": order["status"],
        "already_refunded": order["refunded"],
    }


def get_refund_policy(category: str) -> dict:
    """Return the refund policy for a product category.

    Args:
        category: The product category, e.g. "electronics" or "clothing".
            Unknown categories fall back to the default policy.

    Returns:
        A dict with the refund window in days, the conditions that make an item
        always eligible (e.g. "damaged"), and a human-readable note.
    """
    policy = REFUND_POLICY.get(category.strip().lower(), REFUND_POLICY["default"])
    return {
        "category": category,
        "window_days": policy["window_days"],
        "always_eligible_if": policy["always_eligible_if"],
        "notes": policy["notes"],
    }


def issue_refund(order_id: str, amount: float) -> dict:
    """Issue a refund for an order. THIS IS THE WRITE ACTION.

    Only call this after confirming, via get_order_status and get_refund_policy,
    that the order exists, has not already been refunded, and is eligible under
    policy. Never refund more than the order's price.

    Args:
        order_id: The order to refund, e.g. "A1001".
        amount: The amount to refund in dollars. Must not exceed the price paid.

    Returns:
        A dict confirming the refund, or an error describing why it was refused.
    """
    order = ORDERS.get(order_id.strip().upper())
    if order is None:
        return {"success": False, "error": f"No order found with ID '{order_id}'."}
    if order["refunded"]:
        return {
            "success": False,
            "error": f"Order {order['order_id']} has already been refunded. Not issuing a second refund.",
        }
    if amount > order["price"]:
        amount_str = format(amount, ".2f")
        price_str = format(order["price"], ".2f")
        return {
            "success": False,
            "error": (
                "Refund amount of USD " + amount_str
                + " exceeds the order price of USD " + price_str + "."
            ),
        }
    # In a real system this would write to a ledger. Here we just confirm.
    amount_str = format(round(amount, 2), ".2f")
    return {
        "success": True,
        "order_id": order["order_id"],
        "refund_amount": round(amount, 2),
        "eta_business_days": "5-7",
        "message": "Refund of USD " + amount_str + " approved for order " + order["order_id"] + ".",
    }


# Wrap the plain functions as ADK tools.
# FunctionTool is the core mechanism used to transform standard source code functions into custom
# tools that an AI agent can understand and execute
get_order_status_tool = FunctionTool(func=get_order_status)
get_refund_policy_tool = FunctionTool(func=get_refund_policy)
issue_refund_tool = FunctionTool(func=issue_refund)

ALL_TOOLS = [get_order_status_tool, get_refund_policy_tool, issue_refund_tool]


In [ ]:
%%writefile refund_agent/agent.py
"""The refund-support agent.

This is the single agent we evaluate during the workshop

The core of this agent is in the `instructions` and its `tools`.
"""

import os

from google.adk.agents import Agent
from google.genai import types

from .tools import ALL_TOOLS

# The LLM model used by the Agent
MODEL = os.environ.get("WORKSHOP_MODEL", "gemini-3.1-flash-lite")

INSTRUCTION = """\
You are a customer-support agent for an online store. You handle refund requests.

Follow this procedure for EVERY refund request, in this exact order:

1. Identify the order. Call `get_order_status` with the order ID.
   - If the order is not found, do NOT guess. Ask the customer to double-check
     their order ID. Do not call any other tool.
   - If the order has already been refunded, tell the customer and do NOT issue
     another refund.

2. Check eligibility. Call `get_refund_policy` for the order's category BEFORE
   issuing any refund. An item is eligible if either:
     - it is within the policy's refund window (today minus the order date is
       less than or equal to window_days), OR
     - the customer says it arrived damaged or defective and that condition is
       in the policy's "always_eligible_if" list.
     - if the item is NOT eligible, politely decline and explain why. Do NOT call
     `issue_refund`.

3. Issue the refund. Only if the order exists, has not been refunded, and is
   eligible, call `issue_refund` with the order ID and the price paid. Never
   refund more than the price.

When you reply to the customer, always:
  - state the refund amount and the expected timeline if a refund was issued,
  - give a clear, specific reason for the decision,
  - keep a warm, professional tone,
  - never invent policy details that the tools did not return.

Today's date is 2nd July 2026.
"""
# home work for the agent - replace the instruction for today's date with a tool

root_agent = Agent(
    name="refund_agent",
    model=MODEL,
    description="Handles customer refund requests for an online store.",
    instruction=INSTRUCTION,
    tools=ALL_TOOLS
)


In [10]:
%%writefile refund_agent/__init__.py
"""Refund-support agent package.

ADK discovers the agent through this import: `adk web` and `adk eval` look for a
module that exposes `root_agent`.
"""

from . import agent
from .agent import root_agent

__all__ = ["agent", "root_agent"]


Overwriting refund_agent/__init__.py


### See it run (this replaces `adk web`)

`adk web` is ADK'S built in browser based developer interface. For Colab we run the agent in code and
print its **tool trajectory** and final reply. This is the live behavior the evals will assert on.

> **Important — picking up your edits.** When you change `refund_agent/agent.py` later (the
> "make it break" exercise) and re-run its `%%writefile` cell, the *file* changes but the kernel
> still holds the **old imported module in memory**. So `show_run` below reloads the agent from
> disk on every call — that's why your edits take effect here without a runtime restart. (The
> `adk eval` cells never have this problem: each runs in a fresh subprocess.)

In [11]:
import sys
from google.adk.runners import InMemoryRunner
from google.genai import types

def _load_root_agent():
    """Re-import refund_agent from disk so edits to its files are picked up."""
    for name in [m for m in sys.modules if m == "refund_agent" or m.startswith("refund_agent.")]:
        del sys.modules[name]
    import refund_agent
    return refund_agent.root_agent

async def show_run(query, user_id="u"):
    """Run the agent on one message and print the user input, the tool
    trajectory, and the final agent reply — clearly separated."""
    runner = InMemoryRunner(agent=_load_root_agent(), app_name="refund_agent")
    session = await runner.session_service.create_session(
        app_name="refund_agent", user_id=user_id)
    content = types.Content(role="user", parts=[types.Part(text=query)])

    tool_calls, final = [], "(no final response)"
    async for event in runner.run_async(
            user_id=user_id, session_id=session.id, new_message=content):
        if event.content and event.content.parts:
            for p in event.content.parts:
                if getattr(p, "function_call", None):
                    args = ", ".join(f"{k}={v!r}" for k, v in dict(p.function_call.args).items())
                    tool_calls.append(f"{p.function_call.name}({args})")
        if event.is_final_response() and event.content and event.content.parts:
            text = "".join(p.text for p in event.content.parts if getattr(p, "text", None))
            if text:
                final = text.strip()

    print("=" * 72)
    print(">>> USER:  " + query)
    print("-" * 72)
    print("    tool calls:")
    for t in tool_calls:
        print("      - " + t)
    print("-" * 72)
    print("<<< AGENT: " + final)
    print("=" * 72)
    # no return value, so the cell doesn't echo a raw tuple

await show_run("Hi, I'd like a refund for my order A1001.")

>>> USER:  Hi, I'd like a refund for my order A1001.
------------------------------------------------------------------------
    tool calls:
      - get_order_status(order_id='A1001')
      - get_refund_policy(category='electronics')
      - issue_refund(amount=79.99, order_id='A1001')
------------------------------------------------------------------------
<<< AGENT: Hello! I would be happy to help you with that refund request for your order A1001 (Wireless headphones).

I have processed your refund of $79.99. Since your order was placed on June 25th, it falls well within our 30-day return policy for electronics. You should see the funds returned to your original payment method within 5-7 business days.

If there is anything else I can assist you with, please feel free to ask!


Try a few more — each takes a different path:

In [12]:
await show_run("Please refund order A1002.")        # outside the window -> should decline
await show_run("I want a refund for order A1004.")    # unknown order -> should ask, not refund

>>> USER:  Please refund order A1002.
------------------------------------------------------------------------
    tool calls:
      - get_order_status(order_id='A1002')
      - get_refund_policy(category='electronics')
------------------------------------------------------------------------
<<< AGENT: I'm sorry, but I am unable to process a refund for your order A1002.

The refund policy for electronics allows for returns within 30 days of the purchase date, or at any time if the item is damaged or defective. As your order was placed on April 1st, 2026, it is outside of the 30-day refund window. Since you did not mention that the item is damaged or defective, I cannot authorize a refund at this time.
>>> USER:  I want a refund for order A1004.
------------------------------------------------------------------------
    tool calls:
      - get_order_status(order_id='A1004')
------------------------------------------------------------------------
<<< AGENT: I'm sorry, but I couldn't fin

## 2. Trajectory evaluation

A trajectory eval asserts the agent took the **right steps** — the right tool calls in the right
order — not just that the text looks plausible. This is what catches dangerous behavior (like
refunding *before* checking the policy).

We write the golden eval set (5 cases covering every branch) and a config that requires a perfect
tool-trajectory score, then run `adk eval`.

In practise we use `adk web` to generate the evaltest file instead of writing the file by hand. The most important things are `user_content`, `final_response` and `tools_use`

In [ ]:
%%writefile eval/trajectory.evalset.json
{
  "eval_set_id": "refund_agent_golden",
  "name": "Refund agent — golden trajectory set",
  "description": "Five cases covering every branch the refund agent can take. This is the COMPLETE reference set used for the trajectory demo and for CI. Attendees build a subset of this themselves in exercises/01_trajectory.",
  "eval_cases": [
    {
      "eval_id": "A1001_valid_within_window",
      "conversation": [
        {
          "invocation_id": "inv_a1001",
          "user_content": {
            "parts": [{ "text": "Hi, I'd like a refund for my order A1001." }]
          },
          "final_response": {
            "role": "model",
            "parts": [
              {
                "text": "I've approved a $79.99 refund for order A1001 (Wireless headphones). It will appear on your original payment method within 5-7 business days. Is there anything else I can help with?"
              }
            ]
          },
          "intermediate_data": {
            "tool_uses": [
              { "name": "get_order_status", "args": { "order_id": "A1001" } },
              { "name": "get_refund_policy", "args": { "category": "electronics" } },
              { "name": "issue_refund", "args": { "order_id": "A1001", "amount": 79.99 } }
            ]
          }
        }
      ],
      "session_input": { "app_name": "refund_agent", "user_id": "u1", "state": {} }
    },
    {
      "eval_id": "A1002_outside_window_declined",
      "conversation": [
        {
          "invocation_id": "inv_a1002",
          "user_content": {
            "parts": [{ "text": "Please refund order A1002." }]
          },
          "final_response": {
            "role": "model",
            "parts": [
              {
                "text": "I'm sorry, but order A1002 (Bluetooth speaker) was placed on 2026-04-01, which is outside our 30-day refund window for electronics, so I'm unable to issue a refund. If the item is damaged or defective, let me know and I can take another look."
              }
            ]
          },
          "intermediate_data": {
            "tool_uses": [
              { "name": "get_order_status", "args": { "order_id": "A1002" } },
              { "name": "get_refund_policy", "args": { "category": "electronics" } }
            ]
          }
        }
      ],
      "session_input": { "app_name": "refund_agent", "user_id": "u2", "state": {} }
    },
    {
      "eval_id": "A1003_damaged_refunded",
      "conversation": [
        {
          "invocation_id": "inv_a1003",
          "user_content": {
            "parts": [{ "text": "My order A1003 arrived cracked and damaged. I'd like my money back." }]
          },
          "final_response": {
            "role": "model",
            "parts": [
              {
                "text": "I'm sorry the Ceramic vase arrived damaged. I've approved a $49.99 refund for order A1003, which will reach your original payment method in 5-7 business days."
              }
            ]
          },
          "intermediate_data": {
            "tool_uses": [
              { "name": "get_order_status", "args": { "order_id": "A1003" } },
              { "name": "get_refund_policy", "args": { "category": "home" } },
              { "name": "issue_refund", "args": { "order_id": "A1003", "amount": 49.99 } }
            ]
          }
        }
      ],
      "session_input": { "app_name": "refund_agent", "user_id": "u3", "state": {} }
    },
    {
      "eval_id": "A1004_not_found",
      "conversation": [
        {
          "invocation_id": "inv_a1004",
          "user_content": {
            "parts": [{ "text": "I want a refund for order A1004." }]
          },
          "final_response": {
            "role": "model",
            "parts": [
              {
                "text": "I couldn't find an order with the ID A1004. Could you double-check the order number? It usually appears in your confirmation email."
              }
            ]
          },
          "intermediate_data": {
            "tool_uses": [
              { "name": "get_order_status", "args": { "order_id": "A1004" } }
            ]
          }
        }
      ],
      "session_input": { "app_name": "refund_agent", "user_id": "u4", "state": {} }
    },
    {
      "eval_id": "A1005_already_refunded",
      "conversation": [
        {
          "invocation_id": "inv_a1005",
          "user_content": {
            "parts": [{ "text": "Can you refund order A1005 for me?" }]
          },
          "final_response": {
            "role": "model",
            "parts": [
              {
                "text": "It looks like order A1005 (Running shoes) has already been refunded, so there's nothing further to process on it. If you expected a different order, let me know the ID and I'll check."
              }
            ]
          },
          "intermediate_data": {
            "tool_uses": [
              { "name": "get_order_status", "args": { "order_id": "A1005" } }
            ]
          }
        }
      ],
      "session_input": { "app_name": "refund_agent", "user_id": "u5", "state": {} }
    }
  ]
}


In [ ]:
%%writefile eval/trajectory_config.json
{
  "criteria": {
    "tool_trajectory_avg_score": 1.0
  }
}


Run it. (`adk eval` runs in a fresh process, so it always picks up the latest version of your files.)

In [ ]:
!adk eval refund_agent eval/trajectory.evalset.json --config_file_path eval/trajectory_config.json --print_detailed_results

### Exercise — make it break

Re-open the `refund_agent/agent.py` cell above and
**weaken the instruction** so the agent skips the policy check or sends the wrong order id.

Watch the eval cases  **fail**: the agent might not call the right tools or might not call them in the right order.

Restore the instruction when you're done.

## 3. LLM-as-a-Judge

Trajectory tells you the agent took the right *steps*; it says nothing about whether the reply was
actually *good* — right amount, right timeline, real reason, right tone, nothing made up. You can't
lexically match free-text quality, so you use another model as a judge.

In [ ]:
%%writefile eval/judge.evalset.json
{
  "eval_set_id": "refund_agent_judge_exercise",
  "name": "Section 3 — LLM-as-a-Judge",
  "description": "Two cases whose replies have real quality variance. The agent is run live on each user message, and an LLM judge scores the actual reply against rubrics + a hallucination check.",
  "eval_cases": [
    {
      "eval_id": "A1001_refund_reply_quality",
      "conversation": [
        {
          "invocation_id": "inv_a1001",
          "user_content": {
            "parts": [{ "text": "Hi, I'd like a refund for my order A1001." }]
          },
          "intermediate_data": {
            "tool_uses": [
              { "name": "get_order_status", "args": { "order_id": "A1001" } },
              { "name": "get_refund_policy", "args": { "category": "electronics" } },
              { "name": "issue_refund", "args": { "order_id": "A1001", "amount": 79.99 } }
            ]
          }
        }
      ],
      "session_input": { "app_name": "refund_agent", "user_id": "u1", "state": {} }
    },
    {
      "eval_id": "A1002_decline_reply_quality",
      "conversation": [
        {
          "invocation_id": "inv_a1002",
          "user_content": {
            "parts": [{ "text": "Please refund order A1002." }]
          },
          "intermediate_data": {
            "tool_uses": [
              { "name": "get_order_status", "args": { "order_id": "A1002" } },
              { "name": "get_refund_policy", "args": { "category": "electronics" } }
            ]
          }
        }
      ],
      "session_input": { "app_name": "refund_agent", "user_id": "u2", "state": {} }
    }
  ]
}


In [ ]:
%%writefile eval/judge_config.json
{
  "_comment": "Section 3 — LLM-as-a-Judge. Both metrics run on a Gemini API key (no Vertex). rubric_based_final_response_quality_v1 scores the agent's LIVE reply against specific, checkable rubrics. hallucinations_v1 separately flags any claim in the reply that is NOT grounded in the tool outputs.",
  "criteria": {
    "rubric_based_final_response_quality_v1": {
      "threshold": 0.7,
      "judge_model_options": { "judge_model": "gemini-3.1-flash-lite", "num_samples": 5 },
      "rubrics": [
        { "rubric_id": "states_amount_when_refunded",
          "rubric_content": { "text_property": "If a refund was issued, the response states the exact refund amount in dollars (e.g. $79.99). If no refund was issued, this rubric is satisfied." } },
        { "rubric_id": "states_timeline_when_refunded",
          "rubric_content": { "text_property": "If a refund was issued, the response tells the customer when the money will arrive (e.g. 5-7 business days). If no refund was issued, this rubric is satisfied." } },
        { "rubric_id": "gives_specific_reason",
          "rubric_content": { "text_property": "The response gives a clear, specific reason for the decision (within window, outside the 30-day window, item damaged, order already refunded, or order not found)." } },
        { "rubric_id": "professional_tone",
          "rubric_content": { "text_property": "The response is warm, professional, and appropriate for a customer-facing support reply." } }
      ]
    },
    "hallucinations_v1": {
      "threshold": 0.8,
      "judge_model_options": { "judge_model": "gemini-3.1-flash-lite", "num_samples": 5 }
    }
  }
}


### What we check

**`rubric_based_final_response_quality_v1`** scores the agent's live reply against four explicit,
independently-checkable rubrics:

- `states_amount_when_refunded` — if a refund was issued, the reply states the exact dollar amount.
- `states_timeline_when_refunded` — if a refund was issued, the reply says when the money arrives.
- `gives_specific_reason` — the reply gives a clear, specific reason for the decision.
- `professional_tone` — the reply is warm and professional.

Each rubric is independently true or false, so the score is **explainable** — that's the difference
between a judge you can trust and one you can't.

**`hallucinations_v1`** segments the reply into individual claims and checks each against the
**trusted evidence** (the user's request + the actual tool outputs), catching any invented policy,
refund window, or amount the tools never returned. So the rubrics check *"did the reply include what
a good answer needs"* and hallucinations checks *"did it state anything the tools didn't support."*

Underneath: the agent is **run live** on each message (A1001 → refund, A1002 → decline), and the
judge scores the *actual* reply — it ignores any reference answer in the eval set.

In [ ]:
!adk eval refund_agent eval/judge.evalset.json --config_file_path eval/judge_config.json --print_detailed_results

### Read the output

For each case you get the `prompt`, the `actual_response`, the per-rubric scores **with the judge's
reasoning**, and the hallucination score. Read the reasoning — that's how you sanity-check the judge
itself. `num_samples` is 5 so the score is stable across runs; a judge that disagrees with itself
run-to-run is its own kind of broken.

## 4. Eval as an engineering discipline (CI)


In the workshop repo, `eval/test_refund_agent.py` wraps the golden set as a `pytest` test, and
`.github/workflows/eval.yml` runs it on every push/PR

## Appendix — which metrics need Vertex AI vs an API key

Most ADK metrics run on a plain Gemini API key. A few require a Vertex AI (GCP) project and will
fail on an API key with `API_KEY_SERVICE_BLOCKED`:

**Vertex-only:** `safety_v1`, `response_evaluation_score`, `multi_turn_task_success_v1`,
`multi_turn_trajectory_quality_v1`, `multi_turn_tool_use_quality_v1`.

**Work with an API key:** `tool_trajectory_avg_score`, `response_match_score`,
`final_response_match_v2`, `rubric_based_final_response_quality_v1`,
`rubric_based_tool_use_quality_v1`, `hallucinations_v1`, `per_turn_user_simulator_quality_v1`.

For safety on an API key, use a **safety rubric** under `rubric_based_final_response_quality_v1`
instead of `safety_v1`.